# AI 에이전트란?
### AI 에이전트의 정의
>사용자가 정한 목표를 달성하기 위해 AI모델을 활용해 환경과 상호작용하는 시스템

추론+계획+도구를 사용한 실행이 하나로 맞물려 돌아감

### 생성형 AI vs 에이전틱 AI
1. 생성형 AI: 프롬포트에 맞는 결과물을 한번 출력하고 끝

2. 에이전틱 AI: 목표를 달성할 떄까지 추론과 행동을 반복
    - 에이전틱 AI의 핵심 작동방식 - ReAct

### ReAct (생각 → 행동 → 관찰)
> 생각 -> 행동 -> 관찰 -> 생각의 순환반복

(예시)주문이 늦어요 라는 컴플레인을 받은 고객 cs 에이전트
1. 주문상태부터 확인 (생각)

2. ger order(1234)(행동)

3. 4일 지연중 (관찰)

4. 지연 원인 확인 (생각)

... 이런식으로 진행

-> 즉, 관찰 결과에 따라 다음 생각이 달라지는 것이 핵심

목표만 주면 알아서 성공하는 것이 아님, 정해진 범위 안에서 순환을 반복함

나아가 행동단게에서도 모델은 도구를 직접 실행하지 않으며 도구를 ~값으로 불러달라고 요청을 내놓을 뿐 

### 다중 에이전트 : 오케스트레이터 + 서브 에이전트
> 여러 에이전트가 역할을 나눠 함께 일하는 구조

오케스트레이터: 요청을 작은일로 쪼개고 누구에게 맡길지 결정한뒤 결과를 하나로 합침

서브에이전트: 맡은 역할 하나만 

- 역할을 나누는 이유
    - 스스로 실수를 잡기 어려움 

    - 동시에 여러 갈래를 조사

-> 실제 엔트로픽은 이 오케스트레이터에 서브에이전트 구조를 적용해서 단일 에이전트보다 90% 이상 좋은 결과를 냄 (대신 토큰 15배)

-> 주로 따로 조사해서 합칠 수 있는 큰 작업에 사용

# MCP란?
### 도구 호출의 원리 : Function Calling
> 모델이 "이 함수를 이 인자로 불러줘"라는 구조화된 요청을 내놓는 기능

실제 실행과 결과전달은 앱이 담당

1. 앱 → 모델: 질문 + 쓸 수 있는 함수목록

2. 모델: 어떤 함수를 어떤 값으로 부를지 적어서 내놓음

3. 앱: 그 함수를 실제로 실행

4. 앱 → 모델: 실행결과 전달 → 모델이 최종 답 작성

### MCP가 필요한 이유 : n×m → n+m
- 표준이 없을 때: 앱마다 × 도구마다 연동 코드를 따로 작성 → n×m개
    - 도구가 하나 늘 때마다 작업량도 크게 늘어남

- 표준이 생기면: 앱은 MCP 클라이언트 1번, 도구는 MCP 서버 1번만 → n+m개

> MCP: LLM 앱과 외부 데이터, 도구가 통신하는 개방형 표준 프로토콜

-> MCP = AI 앱과 도구 사이의 USB-C, 어떤 앱이든 어떤 도구든 같은 단자로 연결

### MCP의 구조 : Host / Client / Server
1. Host: 사용자가 쓰는 AI 앱 (Claude 데스크톱)
    - LLM을 품고 클라이언트를 만들고 관리, 어떤 서버,도구 호출을 허용할지 결정

2. Client: 호스트 안의 통신 담당
    - 서버 1개당 1개, MCP 규격으로 메시지 전달만

3. Server: 도구,데이터를 MCP 규격으로 내놓는 프로그램 (노션 서버, 깃허브 서버)
    - 가진 기능 선언, 요청이 오면 실제 작업 수행

### MCP 서버의 기본 요소 : Tools / Resources / Prompts
> 서버가 내놓을 수 있는 기능의 종류

1. Tools: 실행할 수 있는 행동 (노션 페이지 만들기, 깃허브 이슈 등록하기)

2. Resources: 읽기 전용 데이터 (파일 내용, 문서, DB 스키마)

3. Prompts: 재사용 가능한 템플릿 (코드리뷰 요청양식, 회의록 요약 양식)

-> RAG가 참고할 문서를 Resources로 넘길 수 있음


# MCP의 한계
### 보안 책임은 누구에게 있는가
> 프로토콜이 아니라 호스트가 책임짐

- 사용자 동의, 서버에 넘길 권한 범위, 허용할 도구 선택 = 모두 호스트 몫

- MCP는 연결하는 법만 정하고 무엇을 허락할지는 정하지 않음

### 도구 오염
연결이 쉬워진 만큼 공격받을 틈도 넓어짐

1. 신뢰할 수 없는 서버를 붙이는 위험
    - MCP 서버는 누구나 만들어 올릴 수 있음

    - 출처 모를 서버 연결 = 출처 모를 프로그램 설치

2. 도구 설명을 통한 주입
> 도구 설명 안에 사용자에겐 안 보이고 모델에게만 보이는 지시를 숨기는 공격

- 모델은 description 전체를 읽고 행동
   
- 모델은 숨긴 지시까지 읽음

3. 너무 넓은 권한 연동의 위험
    - 서버 코드 결함이 아니라 공개·비공개 저장소를 모두 읽고 쓰는 넓은 권한이 문제

### 프롬프트 주입
> 도구가 가져온 내용 안에 지시가 섞여 들어오는 것 → 권한이 넓을수록 피해가 커짐

-> 안전하게 쓰려면: 공식검증된 서버만 연결, 필요한 권한만 주기, 되돌리기 어려운 호출은 사람이 승인, 처음 연결하는 서버는 도구설명 직접확인

# 데이터 구조화
### 왜 구조화가 필요한가
> 에이전틱 AI, 피지컬 AI가 일하려면 결국 구조화된 데이터가 필요.. 

RAG는 단순히 문서를 벡터DB로 만드는 것만으로는 한계가 존재

- Vanilla RAG의 한계 - Top-K 딜레마
    - Top-K: RAG에서 질문과의 유사도가 가장 높은 상위 k개의 문서 조각만 선택하여 LLM에 전달하는 검색방식

    - k가 작으면 → 필요한 정보를 놓침
    - k가 크면 → 관련 없는 정보가 섞임

    -> 적절한 지점을 찾기 어려움

-> 바닐라 RAG의 한계는 근본적으로 청크들 사이의 관계를 표현하지 못하는 것

실제 사내 환경은 더 복잡하기에 구조화되지 않은 데이터를 넣으면 구조화 되지 않은 답변이 나옴 

### 그래프 구조와 팔란티어 온톨로지
> 그래프는 데이터를 노드(점)와 엣지(선)로 표현하는 구조

- 노드: 사물, 존재하는 것 
- 엣지: 노드 사이의 관계 

-> 그래프는 서로 어떻게 연결되어 있는가를 저장

팔란티어 온톨로지: 그래프 구조를 기업 전체에 적용하여, 흩어진 사내 데이터를 하나의 의미 있는 관계망으로 묶은 것

- 온톨로지 구성요소
    1. 객체 타입: 테이블의 스키마 / 객체: 테이블의 한 행 / 객체 세트: 여러 객체의 모음 

    2. 속성: 객체가 가진 특징 / 속성값: 특정 객체가 가지는 속성의 실제 값

    3. 링크 타입: 관계의 스키마 / 링크: 실제 관계의 인스턴스

    4. 액션 타입: 실제 변경과 행동까지 온톨로지 위에서 정의

    5. 역할: 접근 권한을 객체 단위까지 세밀하게 제어, AI 에이전트도 사람처럼 권한을 부여받음

# XAI
## 블랙박스가 왜 문제인가
> 딥러닝 모델은 수십억 개의 파라미터가 복잡하게 얽혀 있어 사람이 직접 읽을 수 없음 → 입력과 출력 사이 과정은 사실상 블랙박스

- 단순히 "알면 더 좋은" 수준이 아닌 이유 => 규제 & 디버깅
    - 규제: 의료(FDA는 진단 AI에 근거 요구), 자율주행(사고시 책임 소재)

    - 디버깅: 모델이 잘못 학습했을 때 어디가 문제인지 찾으려면 내부를 봐야 함

## 기존 해결 시도들
1. 회귀 분석: 완전한 화이트박스, 계수를 직접 읽을 수 있음
    - 한계: 설명은 완벽한데 성능이 너무 낮음

2. SHAP / LIME: 사후적 설명, 외부에서 어떤 입력 특성이 결과에 얼마나 기여했는지 근사 계산
    - 한계: 역으로 추정하는 방법이라 진짜 이유가 아닌 그렇게 보이는 것일 수 있음

3. LLM + RAG: 블랙박스로 블랙박수 설명하기
    - 이유를 GPT한테 물어보자는 접근법
    - 한계: 블랙박스가 블랙박스를 설명하는 셈, GPT 설명이 실제 내부 연산과 일치한다는 보장이 없음

## Mechanistic Interpretability
> 모델이 무엇을 출력하는가가 아니라, 모델이 어떻게 그 답에 도달하는가를 이해하자

가중치, 뉴런, 어텐션 헤드를 직접 분석해서 회로 수준까지 역공학하는 연구 분야

- 일반 XAI: 외부에서 관찰 (집 창문으로 들여다보기)

- Mechanistic Interpretability: 내부를 직접 해부 (집 구조도를 직접 그리기)

핵심 기법 : 활성값 패칭 -> 모델 내부의 어느 위치가 답을 결정하는지 찾음

1. 입력 두개 준비

2. B를 처리하는 도중, 특정 위치의 값을 A의 값으로 바꿔 끼움

3. 출력이 어떻기 바뀌는지 봄

-> 정상 기기의 부품을 하나씩 바꿔 끼워 보며 어떤 부품이 결과를 좌우하는지 찾아내는 것과 같음

## 핵심 개념 3가지
1. Features: 모델이 입력에서 알아챈 개념
    - 개별 뉴런 또는 방향 벡터에 저장, 입력에 그 개념이 등장하면 해덩 뉴련이 켜짐

2. Circuits: 여러 뉴런/어텐션 헤드가 연결되어 특정 기능을 수행하는 서브네트워크
    - 회로를 발견하면 조작이 이론적으로 가능

3. Superposition: 하나의 뉴런이 여러 개념을 중첩해서 저장
    - 뉴런 하나를 보면 여러 특징이 조금씩 걸쳐 보이게 됨

    - "뉴런 하나 = 개념 하나"가 아니라 해석이 어려움 → Sparse Autoencoder(SAE)로 해결